# Coffee Coop — 01. Construcción del Source

Este notebook arma la carpeta `data/source` con los catálogos que usa el generador de datos sintéticos:

| Archivo | Hojas | Origen |
|---|---|---|
| `CatalogoDatos.xlsx` | `DIVIPOLA-Departamentos`, `DIVIPOLA-Municipios` | Catálogo DIVIPOLA (DANE) del Source original |
| | `DEPARTAMENTOS-CAFETEROS` | Departamentos productores y su peso relativo |
| | `VARIEDADES`, `CERTIFICACIONES` | Catálogos del caso |
| | `NOMBRES-FINCAS`, `OBSERVACIONES` | Textos para fincas y análisis de calidad |
| `people.xlsx` | `NOMBRES`, `APELLIDOS`, `DOMINIOS-CORREO`, `PREFIJOS-CELULAR` | Listas para generar personas |
| `addresses.xlsx` | `CODIGOS_POSTALES_BARRIOS_VEREDA` | Copia del Source original (veredas y límites por municipio) |

Solo se ejecuta una vez, o cuando quieras cambiar las listas. Los pesos (`weight`) se pueden editar directamente en el Excel.

In [ ]:
import os
import shutil
import pandas as pd

## 1. Rutas

- `ORIGINAL_SOURCE_PATH`: carpeta donde están `CatalogoDatos.xlsx` y `addresses.xlsx` originales (los de Smart Health).
- `SOURCE_PATH`: carpeta `data/source` del caso Coffee Coop.

En Colab, monta el Drive y cambia `MAIN_PATH`.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

MAIN_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # content/coffee-coop
ORIGINAL_SOURCE_PATH = os.path.join(MAIN_PATH, 'data', 'source-original')
SOURCE_PATH = os.path.join(MAIN_PATH, 'data', 'source')
os.makedirs(SOURCE_PATH, exist_ok=True)
print(MAIN_PATH)

## 2. DIVIPOLA (departamentos y municipios)

In [ ]:
original_catalog = os.path.join(ORIGINAL_SOURCE_PATH, 'CatalogoDatos.xlsx')
df_deptos = pd.read_excel(original_catalog, sheet_name='DIVIPOLA-Departamentos', dtype='string')
df_municipios = pd.read_excel(original_catalog, sheet_name='DIVIPOLA-Municipios', dtype='string')
print(df_deptos.shape, df_municipios.shape)

## 3. Departamentos cafeteros

Solo estos departamentos tendrán caficultores, fincas y puntos de compra. El `weight` es un peso **sintético** que controla cuántas fincas caen en cada departamento (no es la estadística oficial). La latitud y longitud son el centro aproximado de la zona cafetera y se usan para ubicar las fincas.

In [ ]:
coffee_departments = [
    # code, name, weight, approximate latitude / longitude of the coffee zone
    ('41', 'HUILA', 17, 2.30, -75.80), ('05', 'ANTIOQUIA', 14, 6.10, -75.70), ('73', 'TOLIMA', 12, 4.10, -75.30),
    ('19', 'CAUCA', 10, 2.40, -76.70), ('17', 'CALDAS', 7, 5.20, -75.40), ('52', 'NARIÑO', 6, 1.40, -77.20),
    ('54', 'NORTE DE SANTANDER', 6, 7.80, -72.80), ('66', 'RISARALDA', 5, 5.00, -75.90),
    ('76', 'VALLE DEL CAUCA', 5, 4.10, -76.20), ('68', 'SANTANDER', 5, 6.60, -73.30),
    ('25', 'CUNDINAMARCA', 4, 4.80, -74.40), ('63', 'QUINDIO', 3, 4.50, -75.70), ('20', 'CESAR', 2, 10.20, -73.40),
    ('47', 'MAGDALENA', 1.5, 10.90, -73.90), ('15', 'BOYACÁ', 1.5, 5.70, -73.40), ('44', 'LA GUAJIRA', 0.5, 10.90, -73.10),
    ('18', 'CAQUETÁ', 0.5, 1.70, -75.70), ('50', 'META', 0.5, 3.50, -74.00),
]
df_coffee_depts = pd.DataFrame(coffee_departments, columns=['department_code', 'department_name', 'weight', 'latitude', 'longitude'])
df_coffee_depts

## 4. Variedades y certificaciones

In [ ]:
varieties = [
    # name, species, is_rust_resistant, weight, description
    ('Castillo', 'arabica', True, 40, 'Variedad colombiana resistente a la roya, la más sembrada del país.'),
    ('Colombia', 'arabica', True, 12, 'Variedad compuesta resistente a la roya, antecesora de Castillo.'),
    ('Caturra', 'arabica', False, 14, 'Mutación de Borbón de porte bajo y alta productividad.'),
    ('Cenicafé 1', 'arabica', True, 8, 'Variedad de porte bajo con resistencia a la roya.'),
    ('Típica', 'arabica', False, 4, 'Variedad tradicional de porte alto.'),
    ('Borbón', 'arabica', False, 3, 'Variedad tradicional de buena calidad en taza.'),
    ('Tabi', 'arabica', True, 3, 'Variedad de porte alto resistente a la roya.'),
    ('Geisha', 'arabica', False, 2, 'Variedad de especialidad con perfil floral.'),
    ('Borbón Rosado', 'arabica', False, 1.5, 'Variedad de especialidad de fruto rosado.'),
    ('Maragogipe', 'arabica', False, 1, 'Variedad de grano grande.'),
    ('Pacamara', 'arabica', False, 1, 'Cruce de Pacas y Maragogipe, grano grande.'),
    ('Robusta', 'robusta', True, 0.5, 'Especie Coffea canephora, de mayor contenido de cafeína.'),
]
df_varieties = pd.DataFrame(varieties, columns=['variety_name', 'species', 'is_rust_resistant', 'weight', 'description'])

certifications = [
    # name, issuing body, premium per kg (COP, synthetic), weight
    ('Rainforest Alliance', 'Rainforest Alliance', 300, 30),
    ('Fairtrade', 'Fairtrade International', 400, 20),
    ('4C', '4C Services', 150, 25),
    ('Orgánico', 'Ente certificador orgánico', 800, 8),
    ('C.A.F.E. Practices', 'Starbucks / SCS Global Services', 250, 10),
    ('Nespresso AAA', 'Nespresso / Rainforest Alliance', 350, 5),
    ('Bird Friendly', 'Smithsonian Migratory Bird Center', 500, 2),
]
df_certifications = pd.DataFrame(certifications, columns=['certification_name', 'issuing_body', 'premium_per_kg', 'weight'])
df_varieties

## 5. Textos: nombres de fincas y observaciones de calidad

In [ ]:
farm_names = [
    'La Esperanza', 'El Paraíso', 'Buenavista', 'La Palma', 'Villa Rosa', 'El Mirador', 'San José', 'La Primavera',
    'El Diamante', 'La Floresta', 'Santa Elena', 'El Recreo', 'La Cabaña', 'Bellavista', 'El Porvenir', 'La Fortuna',
    'Las Brisas', 'El Edén', 'La Pradera', 'Monteverde', 'El Encanto', 'La Montaña', 'Los Naranjos', 'El Cedro',
    'La Ceiba', 'Las Delicias', 'El Rosal', 'La Cumbre', 'Altamira', 'El Retiro', 'La Granja', 'Los Alpes',
    'El Guamo', 'La Aurora', 'Santa Bárbara', 'San Luis', 'La Victoria', 'El Jardín', 'Las Palmas', 'La Loma',
    'El Tesoro', 'Villa Hermosa', 'Los Pinos', 'El Silencio', 'La Selva', 'El Vergel', 'Miraflores', 'La Unión',
    'El Manantial', 'La Cascada', 'El Arrayán', 'Las Acacias', 'La Isabela', 'El Caney', 'Monte Alto', 'La Holanda',
    'El Brillante', 'Las Margaritas', 'La Argentina', 'El Bosque', 'Agua Bonita', 'La Siberia', 'El Placer',
    'San Rafael', 'La Reserva', 'El Triunfo', 'Las Mercedes', 'La Soledad', 'El Higuerón', 'La Estrella',
    'Los Guaduales', 'El Nogal', 'La Gaitana', 'El Mango', 'La Quebrada', 'Bella Vista Alta', 'El Oasis',
    'Santa Clara', 'La Divisa', 'El Recuerdo', 'Las Orquídeas', 'Mi Ranchito', 'El Refugio', 'La Florida',
    'San Antonio', 'La Milagrosa', 'El Danubio', 'Los Arrayanes', 'La Carolina', 'El Cafetal', 'Tierra Linda',
    'El Horizonte', 'La Gloria', 'Cafetal Viejo', 'El Guayabo', 'La Cristalina', 'El Mesón', 'Los Laureles',
]
df_farm_names = pd.DataFrame({'farm_name': farm_names})

observations = [
    # text, applies_to: accepted / rejected / any
    ('Grano uniforme, buen secado.', 'accepted'),
    ('Café de buena presentación, sin olores extraños.', 'accepted'),
    ('Leve presencia de grano brocado dentro de lo tolerable.', 'accepted'),
    ('Muestra limpia, color homogéneo.', 'accepted'),
    ('Buen rendimiento en trilla.', 'accepted'),
    ('Taza limpia con notas dulces.', 'accepted'),
    ('Humedad en el límite superior, se recomienda mejorar el secado.', 'accepted'),
    ('Café con humedad alta, requiere más secado.', 'rejected'),
    ('Café sobresecado, grano quebradizo.', 'rejected'),
    ('Alto porcentaje de grano negro y vinagre.', 'rejected'),
    ('Olor a fermento en la muestra.', 'rejected'),
    ('Presencia de grano mohoso.', 'rejected'),
    ('Exceso de impurezas y pasilla.', 'rejected'),
    ('Muestra tomada por duplicado.', 'any'),
    ('Caficultor solicita contramuestra.', 'any'),
]
df_observations = pd.DataFrame(observations, columns=['observation', 'applies_to'])

## 6. Personas: nombres, apellidos, dominios y prefijos

In [ ]:
male_names = [
    'Juan', 'Carlos', 'José', 'Luis', 'Jorge', 'Andrés', 'Pedro', 'Miguel', 'Diego', 'Jairo', 'Hernando', 'Alberto',
    'Fabio', 'Óscar', 'Álvaro', 'Gustavo', 'Héctor', 'Rubén', 'Rafael', 'Ramiro', 'Orlando', 'Wilson', 'Édgar',
    'Javier', 'Fernando', 'Mauricio', 'Camilo', 'Sebastián', 'Santiago', 'Alejandro', 'Daniel', 'David', 'Felipe',
    'Julián', 'Nelson', 'Arturo', 'Hugo', 'Guillermo', 'Rodrigo', 'Gilberto', 'Libardo', 'Aníbal', 'Efraín',
    'Ancízar', 'Heriberto', 'Abelardo', 'Reinaldo', 'Jaime', 'Mario', 'Manuel', 'Francisco', 'Antonio', 'Ricardo',
    'Germán', 'Iván', 'Wilmer', 'Yeison', 'Brayan', 'Cristian', 'Esteban', 'Mateo', 'Nicolás', 'Samuel', 'Emilio',
]
female_names = [
    'María', 'Ana', 'Luz', 'Martha', 'Gloria', 'Rosa', 'Carmen', 'Blanca', 'Teresa', 'Gladys', 'Amparo', 'Esperanza',
    'Fanny', 'Nubia', 'Stella', 'Olga', 'Beatriz', 'Patricia', 'Claudia', 'Sandra', 'Adriana', 'Diana', 'Liliana',
    'Marleny', 'Yolanda', 'Rocío', 'Myriam', 'Elsa', 'Inés', 'Lucía', 'Alba', 'Carolina', 'Paola', 'Andrea',
    'Natalia', 'Juliana', 'Valentina', 'Daniela', 'Camila', 'Laura', 'Sofía', 'Mariana', 'Alejandra', 'Yuliana',
    'Leidy', 'Yesenia', 'Erika', 'Viviana', 'Marcela', 'Lorena', 'Mónica', 'Ángela', 'Ruth', 'Cecilia', 'Mercedes',
    'Dora', 'Edilma', 'Aura', 'Jazmín', 'Sara', 'Isabella', 'Gabriela', 'Manuela', 'Luisa',
]
surnames = [
    'Rodríguez', 'Gómez', 'González', 'Martínez', 'García', 'López', 'Hernández', 'Sánchez', 'Ramírez', 'Pérez',
    'Díaz', 'Muñoz', 'Rojas', 'Moreno', 'Jiménez', 'Vargas', 'Castro', 'Gutiérrez', 'Álvarez', 'Ruiz', 'Ortiz',
    'Torres', 'Suárez', 'Romero', 'Herrera', 'Valencia', 'Quintero', 'Giraldo', 'Restrepo', 'Ospina', 'Cardona',
    'Osorio', 'Arango', 'Zapata', 'Marín', 'Londoño', 'Salazar', 'Mejía', 'Correa', 'Henao', 'Agudelo', 'Montoya',
    'Duque', 'Betancur', 'Arias', 'Bedoya', 'Cárdenas', 'Mora', 'Ríos', 'Parra', 'Peña', 'Rincón', 'Carrillo',
    'Jaimes', 'Villamizar', 'Contreras', 'Rangel', 'Sandoval', 'Pabón', 'Ortega', 'Flórez', 'Duarte', 'Calderón',
    'Acevedo', 'Bermúdez', 'Cáceres', 'Caicedo', 'Cuéllar', 'Perdomo', 'Trujillo', 'Polanía', 'Cabrera', 'Losada',
    'Tovar', 'Silva', 'Medina', 'Ramos', 'Castillo', 'Guerrero', 'Rivera', 'Delgado', 'Molina', 'Vélez', 'Ibarra',
    'Burbano', 'Bastidas', 'Erazo', 'Benavides', 'Narváez', 'Chamorro', 'Lasso', 'Mosquera', 'Valencia', 'Bonilla',
    'Galvis', 'Pinzón', 'Barrera', 'Niño', 'Vera', 'Lozano', 'Prada', 'Serrano', 'Uribe', 'Zuluaga',
]
df_names = pd.DataFrame(
    [(n, 'M') for n in male_names] + [(n, 'F') for n in female_names],
    columns=['first_name', 'gender'])
df_surnames = pd.DataFrame({'surname': sorted(set(surnames))})
df_domains = pd.DataFrame({'domain': ['gmail.com', 'hotmail.com', 'outlook.com', 'yahoo.com', 'gmail.com', 'gmail.com',
                                      'hotmail.com', 'live.com', 'icloud.com']})
df_prefixes = pd.DataFrame({'prefix': ['300', '301', '302', '304', '305', '310', '311', '312', '313', '314', '315',
                                       '316', '317', '318', '319', '320', '321', '322', '323', '324', '350']})
print(len(df_names), len(df_surnames))

## 7. Guardar el Source

In [ ]:
catalog_file = os.path.join(SOURCE_PATH, 'CatalogoDatos.xlsx')
with pd.ExcelWriter(catalog_file) as writer:
    df_deptos.to_excel(writer, sheet_name='DIVIPOLA-Departamentos', index=False)
    df_municipios.to_excel(writer, sheet_name='DIVIPOLA-Municipios', index=False)
    df_coffee_depts.to_excel(writer, sheet_name='DEPARTAMENTOS-CAFETEROS', index=False)
    df_varieties.to_excel(writer, sheet_name='VARIEDADES', index=False)
    df_certifications.to_excel(writer, sheet_name='CERTIFICACIONES', index=False)
    df_farm_names.to_excel(writer, sheet_name='NOMBRES-FINCAS', index=False)
    df_observations.to_excel(writer, sheet_name='OBSERVACIONES', index=False)

people_file = os.path.join(SOURCE_PATH, 'people.xlsx')
with pd.ExcelWriter(people_file) as writer:
    df_names.to_excel(writer, sheet_name='NOMBRES', index=False)
    df_surnames.to_excel(writer, sheet_name='APELLIDOS', index=False)
    df_domains.to_excel(writer, sheet_name='DOMINIOS-CORREO', index=False)
    df_prefixes.to_excel(writer, sheet_name='PREFIJOS-CELULAR', index=False)

# addresses.xlsx se copia tal cual: trae las veredas por municipio
shutil.copy(os.path.join(ORIGINAL_SOURCE_PATH, 'addresses.xlsx'), os.path.join(SOURCE_PATH, 'addresses.xlsx'))

print('✅ Source listo en', SOURCE_PATH)
print(sorted(os.listdir(SOURCE_PATH)))